# QProgram Vendor Extensions — Registering and Using Them

QProgram core has zero hardware knowledge: it knows about `play`, `measure`, `wait`, `sync`, `set_frequency`, etc., and nothing else. Real hardware (Qblox, Quantum Machines, QDAC, …) has its own bag of operations — `acquire` without a play, marker outputs, trigger I/O, active reset — and those live in **vendor extension packages**.

This notebook walks through:

1. The three pieces a vendor extension package contributes
2. What `import qprogram_qblox` actually does
3. Using vendor operations from user code (both with and without IDE autocomplete)
4. How vendor operations serialize to `.qp` files (`require qblox 0.1` + `qblox.acquire …`)
5. The version-compatibility check on load
6. Combining multiple vendors in one program

If you haven't read `basics.ipynb` yet, start there — this notebook assumes you know what `QProgram`, buses, variables, and the `.qp` format are.

In [ ]:
import qprogram as qp

## 1. What a vendor extension contributes

A vendor extension is a separate Python package that depends on `qprogram`. Conceptually it adds **three orthogonal layers** on top of the core:

| Layer | What it is | What it enables |
|---|---|---|
| **Operation classes** | `Operation` subclasses (e.g. `Acquire`, `SetMarkers`) holding the data that lives in the AST | The new operations exist as typed nodes in the program tree |
| **`VendorNamespace`** | A class with one typed method per operation (e.g. `acquire(bus, weights, save_adc=False)`) | `program.qblox.acquire(...)` works at runtime |
| **Registry entries** | `register_vendor`, `register_vendor_operation`, `register_vendor_version` calls in the package's `__init__.py` | The namespace, operations, and version are visible to base `QProgram`, the `.qp` writer, and the `.qp` parser |

Optionally, the package also ships:

- a **typed mixin** (`QbloxMixin`) — adds `.qblox` as a `@property` so IDE autocomplete works on the Python side
- a **pre-combined `QProgram`** — `from qprogram_qblox import QProgram` gives you the base class with the mixin already applied

Look at the qblox package's `__init__.py` for the canonical example — it's about 30 lines of registration calls and re-exports.

## 1a. Vendor operations are richer than "one sequencer instruction"

QProgram core puts hardware and software operations on equal footing — the compiler decides at execution time which is which (see the *Hardware vs Software Execution* section of the DSL spec). Vendor extensions inherit that freedom: a vendor namespace can expose operations that fall into any of three broadly different shapes:

| Shape | What it is | qblox examples |
|---|---|---|
| **Simple 1-1 hardware op** | Maps to a single sequencer instruction; the writer emits roughly what the hardware will execute. | `set_markers`, `set_trigger`, `wait_trigger`, `acquire` |
| **Complex orchestration** | Bundles several primitive steps into a higher-level intent. The vendor library lowers it to multiple sequencer instructions at compile time. | `active_reset` (measure + conditional reset pulse via the trigger network) |
| **Software-only** | Has no sequencer footprint at all — the platform realizes it via QCoDeS calls, instrument drivers, or other slow-control plumbing at execution time. | `set_acquisition_threshold` (sets a discrimination threshold via QCoDeS) |

From the user's standpoint, all three look identical: `program.qblox.<op>(...)`. From the `.qp` file's standpoint, they all serialize the same way (`qblox.<op_name> <args...>`) — the file is a *description of intent*, not a sequencer dump. The platform-side compiler distinguishes them and dispatches accordingly.

This means a vendor extension is the natural place to put **any** vendor-specific concept whose execution you understand — pulse-level primitives, calibration-side configuration, complex multi-step protocols. You don't need a parallel "setup" API for the slow-control bits and a sequencer API for the fast bits; one namespace covers both.

In [ ]:
from qprogram.buses import BusSchema
from qprogram_qblox import QProgram as QProgramQblox

schema = BusSchema.transmon()
q = schema.q

program = QProgramQblox(label='all three shapes')

# Software-only: sets a QCoDeS threshold at execution time, no sequencer code.
program.qblox.set_acquisition_threshold(q[0].readout, value=0.42)

# Simple 1-1 hardware: configures the sequencer's marker outputs.
program.qblox.set_markers(q[0].drive, mask='0001')

# Complex orchestration: measure + conditional reset pulse.
program.qblox.active_reset(
    bus=q[0].readout,
    waveform='readout_pulse',
    weights='readout_weights',
    control_bus=q[0].drive,
    reset_pulse='pi',
)

import qprogram as qp
print(qp.dumps(program))

#!QProgram 1.0

require qblox 0.1

metadata:
  label: "all three shapes"

schema: transmon

body:
  qblox.set_acquisition_threshold transmon.q[0].readout 0.42
  qblox.set_markers transmon.q[0].drive "0001"
  qblox.active_reset transmon.q[0].readout "readout_pulse" "readout_weights" transmon.q[0].drive "pi" 1 false



All three operations land in the same `body:` section and use the same `qblox.<name>` dot-notation. There is no marker in the file that says "this one is software, this one is sequencer" — that's the platform's job to know.

**Sweeping a software-only parameter inside a loop.** Because `set_acquisition_threshold` accepts an `Expression`, you can sweep it just like a hardware-side parameter. The compiler will run the loop iteration-by-iteration, calling QCoDeS for each step, while still using the sequencer for the embedded pulse operations.

In [ ]:
program = QProgramQblox(label='threshold sweep')
thr = program.variable('thr', label='Discrimination threshold', units='V')

with program.for_loop(thr, start=-0.5, stop=0.5, step=0.01):
    program.qblox.set_acquisition_threshold(q[0].readout, thr)   # software, per iteration
    program.qblox.acquire(q[0].readout, 'weights')               # hardware

print(qp.dumps(program))

#!QProgram 1.0

require qblox 0.1

metadata:
  label: "threshold sweep"

schema: transmon

body:
  var thr label="Discrimination threshold" units="V"

  for thr in range(-0.5, 0.5, 0.01):
    qblox.set_acquisition_threshold transmon.q[0].readout thr
    qblox.acquire transmon.q[0].readout "weights" false



## 2. What `import qprogram_qblox` does

Importing the package is the activation step. At import time `qprogram_qblox/__init__.py` runs three registration calls (plus defines the typed mixin/QProgram for convenience):

```python
# 1. Make `program.qblox` resolve to QbloxNamespace at runtime
QProgram.register_vendor("qblox", QbloxNamespace)

# 2. Tell the .qp parser what version of the qblox protocol we provide
#    (read from package metadata so pyproject.toml is the single source of truth)
register_vendor_version("qblox", __version__)

# 3. Tell the .qp writer & parser how to serialize each operation
register_vendor_operation("qblox", "acquire",       Acquire)
register_vendor_operation("qblox", "set_markers",   SetMarkers)
register_vendor_operation("qblox", "set_trigger",   SetTrigger)
register_vendor_operation("qblox", "wait_trigger",  WaitTrigger)
register_vendor_operation("qblox", "active_reset", ActiveReset)
```

After this runs, *any* `QProgram` instance — including one you constructed before importing qblox — gains a `.qblox` attribute. The registry is class-level state on `QProgram`.

In [ ]:
# Before importing the vendor, .qblox doesn't exist
program = qp.QProgram(label="before")
try:
    program.qblox
except AttributeError as e:
    print('AttributeError:', e)

In [ ]:
import qprogram_qblox  # noqa: F401 — side-effect import: registers the vendor

# Now .qblox is available on every QProgram instance, including this one
ns = program.qblox
print(type(ns).__name__, '— methods:', sorted(m for m in dir(ns) if not m.startswith('_')))

QbloxNamespace — methods: ['acquire', 'active_reset', 'set_acquisition_threshold', 'set_markers', 'set_trigger', 'wait_trigger']


## 3. Using vendor operations — two equivalent ways

### 3.1 Plain `qprogram.QProgram` + side-effect import

After `import qprogram_qblox`, every `QProgram` instance has `.qblox` via the runtime registry. This works but the IDE doesn't know about it — no autocomplete on `qp.qblox.<TAB>`, no type-checker support.

In [ ]:
from qprogram.buses import BusSchema

schema = BusSchema.transmon()
q = schema.q

# Vendor operations typically take string aliases for waveforms — concrete pulse
# shapes get bound later via `program.with_waveforms({...})` at execution time
# (so calibration data lives outside the program). This is the realistic style.

program = qp.QProgram(label="acquire-only readout")

# .qblox resolves through QProgram.__getattr__ → QbloxNamespace
program.qblox.acquire(q[0].readout, "default_weights")
program.qblox.set_markers(q[0].drive, "0001")
print("body has", len(program.body.elements), "ops")


body has 2 ops


### 3.2 Typed `QProgram` from the vendor package

For full IDE autocomplete on `qp.qblox.<method>`, import `QProgram` from the vendor package instead of from `qprogram`. It's the same class, just with `QbloxMixin` already applied:

```python
from qprogram_qblox import QProgram   # has .qblox typed as @property → QbloxNamespace
```

Runtime behaviour is identical — the typing is purely an editor/mypy benefit.

In [ ]:
from qprogram_qblox import QProgram as QProgramQblox

program = QProgramQblox(label="typed-qblox")
program.qblox.acquire(q[0].readout, "default_weights")     # IDE sees this is typed
program.qblox.wait_trigger(q[0].drive, duration=1000, port=2)
print(type(program).__mro__[:3])


(<class 'qprogram_qblox.QProgram'>, <class 'qprogram_qblox.mixin.QbloxMixin'>, <class 'qprogram.qprogram.QProgram'>)


## 4. Serialization: `require <vendor> <version>` and dot-notation operations

A `.qp` file that uses vendor operations declares its dependency in the header. The writer auto-emits one `require` line per used vendor; the parser validates compatibility before parsing the body. Inside the body, vendor operations use **dot notation**: `qblox.acquire`, `qblox.set_markers`, etc.

In [ ]:
program = QProgramQblox(label="vendor demo")
wait_ns = program.variable("wait_ns", label="Wait duration", units="ns")

# One iteration mixes a software-only setup, a complex orchestration, and a few
# simple hardware ops — all under the same `program.qblox.*` namespace.
program.qblox.set_acquisition_threshold(q[0].readout, value=0.42)

with program.for_loop(wait_ns, start=0, stop=1000, step=10):
    program.qblox.wait_trigger(q[0].drive, duration=1000, port=1)
    program.wait(q[0].drive, wait_ns)
    program.qblox.set_markers(q[0].drive, "0001")
    program.qblox.acquire(q[0].readout, "default_weights")

text = qp.dumps(program)
print(text)


#!QProgram 1.0

require qblox 0.1

metadata:
  label: "vendor demo"

schema: transmon

body:
  var wait_ns label="Wait duration" units="ns"

  qblox.set_acquisition_threshold transmon.q[0].readout 0.42
  for wait_ns in range(0, 1000, 10):
    qblox.wait_trigger transmon.q[0].drive 1000 1
    wait transmon.q[0].drive wait_ns
    qblox.set_markers transmon.q[0].drive "0001"
    qblox.acquire transmon.q[0].readout "default_weights" false



Note the `require qblox 0.1` line at the top — that's the **major.minor** of the installed extension's protocol version. The writer reads it from whatever was passed to `register_vendor_version(...)` at import time.

## 5. Round-trip and the version-compatibility check

On load, the parser checks each `require` line against the installed extension's registered version. The rules are:

- **major** must match exactly,
- installed **minor** must be `≥` the file's minor,
- patch is informational only.

So a file with `require qblox 0.1` loads fine against an installed `qblox 0.5`, but not against `qblox 1.0`. If the vendor extension isn't installed at all, the parser fails before parsing the body — giving you a clear error instead of silently dropping operations.

In [ ]:
reloaded = qp.loads(text)
assert qp.dumps(reloaded) == text
print("round-trip OK")

# Confirm the parsed program still uses the Qblox Operation classes
top_kinds = [type(el).__name__ for el in reloaded.body.elements]
print("top-level operations:", top_kinds)

for_loop_block = next(el for el in reloaded.body.elements if hasattr(el, 'elements'))
loop_kinds = [type(el).__name__ for el in for_loop_block.elements]
print("inside the loop    :", loop_kinds)


round-trip OK
top-level operations: ['SetAcquisitionThreshold', 'ForLoop']
inside the loop    : ['WaitTrigger', 'Wait', 'SetMarkers', 'Acquire']


In [ ]:
from qprogram.serialization import ParseError

# Simulate a too-old installed version: a file that asks for qblox 0.99 vs installed 0.1
bad = text.replace('require qblox 0.1', 'require qblox 0.99')
try:
    qp.loads(bad)
except ParseError as e:
    print('rejected:', e)

# Simulate a major-version mismatch
bad2 = text.replace('require qblox 0.1', 'require qblox 1.0')
try:
    qp.loads(bad2)
except ParseError as e:
    print('rejected:', e)

rejected: Line 3: file requires qblox 0.99 or compatible; installed qblox is 0.1.0 — minor version too old
rejected: Line 3: file requires qblox 1.0 (major 1); installed qblox is 0.1.0 (major 0) — major versions must match


## 6. Multiple vendors in one program

When a platform uses operations from more than one vendor (e.g. Qblox for AWG/RF + QDAC for slow DC), each vendor extension registers under its own namespace. Both namespaces coexist; both produce `require` lines on serialization.

For runtime, you don't need to do anything — `program.qblox.*` and `program.qdac.*` both work as soon as both packages are imported. For IDE autocomplete you combine the mixins:

```python
from qprogram import QProgram as BaseQProgram
from qprogram_qblox import QbloxMixin
from qprogram_qdac  import QdacMixin   # hypothetical

class QProgram(QbloxMixin, QdacMixin, BaseQProgram):
    pass

program = QProgram()
program.qblox.acquire(...)   # typed
program.qdac.play(...)       # typed
```

We've only got the qblox vendor available here, but the pattern is open: any number of vendor extensions can stack in the same program, each owning its dot-prefixed namespace.

## 7. What if you want to ship your own vendor?

Mirror the `qprogram-qblox` package layout — four small files:

```
qprogram-myvendor/
└── src/qprogram_myvendor/
    ├── operations.py    # Operation subclasses (the AST nodes)
    ├── namespace.py     # MyVendorNamespace(VendorNamespace) — typed methods
    ├── mixin.py         # MyVendorMixin — @property for IDE autocomplete
    └── __init__.py      # 3-step registration + pre-combined QProgram
```

The `__init__.py` does:

```python
from qprogram.qprogram import QProgram as _BaseQProgram
from qprogram.serialization.registry import register_vendor_operation, register_vendor_version

from qprogram_myvendor.namespace import MyVendorNamespace
from qprogram_myvendor.operations import MyOp1, MyOp2
from qprogram_myvendor.mixin import MyVendorMixin

_BaseQProgram.register_vendor('myvendor', MyVendorNamespace)
register_vendor_version('myvendor', __version__)
register_vendor_operation('myvendor', 'op1', MyOp1)
register_vendor_operation('myvendor', 'op2', MyOp2)

class QProgram(MyVendorMixin, _BaseQProgram):  # pre-combined, typed
    pass
```

Once registered, the operations get free `.qp` serialization (the writer walks `vars(op)` to emit positional args, the parser uses `inspect.signature` to reconstruct), and the version compatibility machinery applies automatically.

---

**Recap.** Vendor extensions plug into QProgram through three orthogonal registries: a runtime namespace (`register_vendor`), a protocol version (`register_vendor_version`), and per-operation serialization mappings (`register_vendor_operation`). Users opt into a vendor with a single import; from that point on `program.<vendor>.<op>(...)` works on any program, and the `.qp` files explicitly declare what they need to be parseable.